# 회의록 청킹, 임베딩, 주제 경계 후보 탐지 후 결과 저장

#### 실험을 통해 결정된 조합 (`02-2_best_boundary_experiment.ipynb`, 20개 회의 기준)

| 항목 | 값 |
|---|---|
| Embedding Method | pooled: 청크 내 발화별 임베딩을 글자 수로 가중합한 뒤 정규화 (`FILLER_MAX_CHARS` 이하 맞장구 제외) |
| Chunk Size / Overlap | 200자 / 100자 (발화 단위로 묶음, 발화 중간에서 자르지 않음) |
| 경계 후보 선택 방식 | threshold: depth > 0 이고 depth ≥ 평균 + `DEPTH_STD_K`·표준편차 |
| Embedding Model | `BAAI/bge-m3` |
| `DEPTH_STD_K` | 0.0 |
| `MERGE_GAP_CHARS` | 400 |
| `MERGE_MAX_SPAN_CHARS` | 800 |

**처리 순서:** ① 전사문 읽기 → ② 청킹 → ③ 발화 임베딩 → 청크 임베딩(pooled) → ④ 블록 유사도 · depth score → 경계 후보 → ⑤ 인접 후보 구간 병합 → 구간별 대표 경계로 회의록 분할 → ⑥ JSON 저장

**구간의 대표 경계:** 병합된 구간 안 후보 중 depth score가 가장 큰 후보의 위치 → 이 발화부터 새 주제 구간이 시작됨

**사용법:** 가장 위 설정 셀의 `INPUT_PATH`에 회의록 전사문(txt)을 지정하고 위에서부터 전체 실행

**출력 JSON 구조** (`OUTPUT_DIR/{meeting_id}_boundary_candidates.json`)

| 키 | 내용 |
|---|---|
| `meeting_id`, `metadata` | 회의록 ID(파일 이름), 상단 머리말(회의 대주제 / 참여자 / 일시) |
| `chunking`, `chunks` | 청킹 설정, 청크별 `text`와 metadata(`chunk_index`, `char_count`, `utterance_start`, `utterance_end`) |
| `detection` | 탐지 설정과 결과 수(depth 임계값, 후보 수, 병합 구간 수, 분할 구간 수) |
| `candidates` | 병합 전 경계 후보 (`boundary_position`: 이 발화부터 새 주제라고 보는 위치) |
| `spans` | 병합된 경계 구간(`span_start`~`span_end`), 묶인 후보, 대표 경계(`boundary_position`) |
| `segments` | **최종 결과** — 대표 경계로 나눈 회의록 구간 (`utterance_start`~`utterance_end`, 발화 목록) |

* 발화 번호(`utterance_index`)는 모두 0부터 셈

In [ ]:
# --------------------------------------------------------------------------------------------------
# 0. 설정
# 사람이 수정하는 값은 이 셀에서만 수정 (회의록을 바꾸려면 INPUT_PATH만 바꾸면 됨)
# 길이는 모두 글자 수(len, 공백·줄바꿈 포함) 기준
# --------------------------------------------------------------------------------------------------
from pathlib import Path

# ── 입력 / 출력 ───────────────────────────────────────
BASE_DIR = Path.cwd().parent                        # playground/chaewon
INPUT_PATH = BASE_DIR / "dataset" / "TS3003c.txt"   # 처리할 회의록 전사문 (txt)
OUTPUT_DIR = BASE_DIR / "outputs" / "boundaries"    # 결과 JSON 저장 폴더 (outputs/는 .gitignore 대상)
SAVE_RESULT = True                                  # False면 저장하지 않고 결과만 확인

# ── 전사문 형식 ───────────────────────────────────────
HEADER_LINES = 3                # 상단 메타데이터 줄 수 (회의 대주제 / 참여자 / 일시) → 청킹 제외
UTTERANCE_SEPARATOR = "\n"      # 청크 안에서 발화를 이어 붙일 구분자

# ── 청킹 ──────────────────────────────────────────────
CHUNK_SIZE = 200                # 청크당 최대 글자 수
CHUNK_OVERLAP = 100             # 인접 청크가 공유할 최대 글자 수 (발화 단위로 맞춤)

# ── 임베딩 ────────────────────────────────────────────
EMBEDDING_MODEL = "BAAI/bge-m3"
DEVICE = None                   # None: 자동 선택 (GPU 있으면 cuda), 또는 "cpu" / "cuda"
EMBED_BATCH_SIZE = 16
EMBED_CACHE_DIR = BASE_DIR / "outputs" / "experiment" / "embeddings" / "bge-m3"
                                 # 실험에서 저장한 문장 단위 임베딩 캐시 (texts.json + vectors.npy)
                                 # 캐시에 있는 발화는 저장된 벡터를 쓰고, 없는 발화만 모델로 임베딩 / None이면 전부 모델로 임베딩
FILLER_MAX_CHARS = 4            # 이 글자 수 이하 발화(네., 음., 좋아요. 등 맞장구)는 청크 임베딩에서 제외

# ── 경계 후보 탐지 ─────────────────────────────────────
BLOCK_SIZE = 2                  # 청크 경계 양쪽에서 평균 내어 비교할 청크 수
DEPTH_STD_K = 0.0               # 후보 임계값 = depth 평균 + K × 표준편차

# ── 인접 후보 구간 병합 ─────────────────────────────────
MERGE_GAP_CHARS = 400           # 이웃한 후보 사이 발화 글자 수가 이 값 이하면 같은 구간으로 묶음
MERGE_MAX_SPAN_CHARS = 800      # 한 구간의 최대 폭 (첫 후보 ~ 마지막 후보 사이 발화 글자 수)

In [ ]:
# --------------------------------------------------------------------------------------------------
# 1. 전사문 읽기 - 머리말(metadata)과 발화 분리, 발화마다 0부터 번호 부여
# --------------------------------------------------------------------------------------------------
import json


def load_transcript(path, header_lines=HEADER_LINES):
    lines = Path(path).read_text(encoding="utf-8-sig").splitlines()
    metadata = {}
    for line in lines[:header_lines]:
        key, _, value = line.partition(":")
        metadata[key.strip()] = value.strip()
    texts = [line.strip() for line in lines[header_lines:] if line.strip()]
    return {
        "meeting_id": Path(path).stem,
        "metadata": metadata,
        "utterances": [{"utterance_index": i, "text": t} for i, t in enumerate(texts)],
    }

In [ ]:
# --------------------------------------------------------------------------------------------------
# 2. 글자 수 기준 발화 단위 청킹
# - 발화를 순서대로 쌓다가 chunk_size를 넘기 직전에 청크를 닫음 (발화 중간에서 자르지 않음)
# - 청크 끝에서 chunk_overlap 이내의 발화를 다음 청크 앞에 다시 포함
# --------------------------------------------------------------------------------------------------
def chunk_utterances(texts, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, sep=UTTERANCE_SEPARATOR):
    assert 0 <= chunk_overlap < chunk_size, "chunk_overlap은 0 이상, chunk_size 미만이어야 합니다."
    chunks = []
    start = 0
    while start < len(texts):
        end = start + 1
        while end < len(texts) and len(sep.join(texts[start:end + 1])) <= chunk_size:
            end += 1
        text = sep.join(texts[start:end])
        chunks.append({
            "text": text,
            "metadata": {
                "chunk_index": len(chunks),
                "char_count": len(text),
                "utterance_start": start,    # 포함
                "utterance_end": end - 1,    # 포함
            },
        })
        if end == len(texts):
            break
        next_start = end                     # 항상 앞으로 진행하도록 start보다 뒤에서 멈춤
        while next_start - 1 > start and len(sep.join(texts[next_start - 1:end])) <= chunk_overlap:
            next_start -= 1
        start = next_start
    return chunks

In [ ]:
# --------------------------------------------------------------------------------------------------
# 3. 임베딩
# - 발화별 임베딩 (캐시에 없는 발화만 모델로 계산)
# - pooled 청크 임베딩: 청크 안 발화 중 맞장구를 빼고, 글자 수로 가중합한 뒤 정규화
# 모든 벡터는 L2 정규화 → 내적 = 코사인 유사도
# --------------------------------------------------------------------------------------------------
import numpy as np
from sentence_transformers import SentenceTransformer

_embed_model = None


def l2norm(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def get_model():
    # 모델은 캐시에 없는 발화가 있을 때만 한 번 불러옴 (BGE-M3 최초 실행 시 약 2.3GB 다운로드)
    global _embed_model
    if _embed_model is None:
        _embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
    return _embed_model


def embed_utterances(texts):
    lookup, cached_vectors = {}, None
    if EMBED_CACHE_DIR is not None and (EMBED_CACHE_DIR / "texts.json").is_file():
        cached_texts = json.loads((EMBED_CACHE_DIR / "texts.json").read_text(encoding="utf-8"))
        cached_vectors = np.load(EMBED_CACHE_DIR / "vectors.npy", mmap_mode="r")
        lookup = {t: i for i, t in enumerate(cached_texts)}

    missing = list(dict.fromkeys(t for t in texts if t not in lookup))   # 중복 없이, 순서 유지
    new = {}
    if missing:
        vectors = get_model().encode(missing, batch_size=EMBED_BATCH_SIZE, normalize_embeddings=True,
                                     convert_to_numpy=True, show_progress_bar=True)
        new = dict(zip(missing, vectors))
    print(f"발화 임베딩: 캐시 사용 {sum(t in lookup for t in texts)}개 / 새로 임베딩 {len(missing)}개 (서로 다른 문장 기준)")
    return l2norm(np.array([cached_vectors[lookup[t]] if t in lookup else new[t] for t in texts], dtype=np.float32))


def pooled_chunk_embeddings(chunks, utt_emb, utt_chars, filler_max_chars=FILLER_MAX_CHARS):
    vectors = []
    for c in chunks:
        m = c["metadata"]
        idx = np.arange(m["utterance_start"], m["utterance_end"] + 1)
        content = idx[utt_chars[idx] > filler_max_chars]
        idx = content if len(content) else idx          # 청크가 전부 맞장구면 그대로 사용
        vectors.append((utt_emb[idx] * utt_chars[idx][:, None]).sum(axis=0))
    return l2norm(np.array(vectors))

In [ ]:
# --------------------------------------------------------------------------------------------------
# 4. 경계 후보 탐지
# gap g = chunk g 와 chunk g+1 사이 (청크가 n개면 gap은 n-1개)
# - 블록 유사도: gap 왼쪽 청크 BLOCK_SIZE개 평균 vs 오른쪽 청크 BLOCK_SIZE개 평균의 코사인 유사도
# - depth score (TextTiling): 유사도 곡선에서 양쪽 봉우리 대비 얼마나 깊게 꺼졌는지
# - 후보: depth > 0 이고 depth ≥ 평균 + DEPTH_STD_K × 표준편차인 gap
# - 위치: gap을 발화 번호로 변환 (두 청크가 겹치는 발화 구간의 가운데, 겹침이 없으면 오른쪽 청크의 첫 발화)
# --------------------------------------------------------------------------------------------------
def block_similarities(emb, block_size=BLOCK_SIZE):
    sims = []
    for g in range(len(emb) - 1):
        left = l2norm(emb[max(0, g - block_size + 1):g + 1].mean(axis=0))
        right = l2norm(emb[g + 1:g + 1 + block_size].mean(axis=0))
        sims.append(float(left @ right))
    return np.array(sims)


def depth_scores(sims):
    depths = []
    for i in range(len(sims)):
        j, left_peak = i, sims[i]
        while j > 0 and sims[j - 1] >= left_peak:
            j -= 1
            left_peak = sims[j]
        j, right_peak = i, sims[i]
        while j < len(sims) - 1 and sims[j + 1] >= right_peak:
            j += 1
            right_peak = sims[j]
        depths.append((left_peak - sims[i]) + (right_peak - sims[i]))
    return np.array(depths)


def select_candidates(depths, std_k=DEPTH_STD_K):
    threshold = depths.mean() + std_k * depths.std()
    gaps = [int(g) for g in np.flatnonzero((depths >= threshold) & (depths > 0))]
    return gaps, float(threshold)


def boundary_position(chunks, g):
    right_start = chunks[g + 1]["metadata"]["utterance_start"]     # 오른쪽 청크가 시작하는 발화 번호
    left_end = chunks[g]["metadata"]["utterance_end"] + 1          # 왼쪽 청크가 끝난 바로 다음 발화 번호
    return (right_start + left_end) // 2

In [ ]:
# --------------------------------------------------------------------------------------------------
# 5. 인접 경계 후보 구간 병합 → 회의록 분할
# - 위치 순으로 후보를 보며, 바로 앞 후보와의 사이 글자 수 ≤ MERGE_GAP_CHARS 이고
#   구간 전체 폭 ≤ MERGE_MAX_SPAN_CHARS이면 같은 구간으로 묶음 (겹침 때문에 같은 전환이 여러 후보로 잡힌 경우)
# - 구간마다 depth score가 가장 큰 후보를 대표 경계로 두고, 대표 경계마다 회의록을 나눔
# --------------------------------------------------------------------------------------------------
def merge_candidates(candidates, utt_chars, gap_chars=MERGE_GAP_CHARS, max_span_chars=MERGE_MAX_SPAN_CHARS):
    groups = []
    for c in sorted(candidates, key=lambda c: c["boundary_position"]):
        if groups:
            first, last = groups[-1][0]["boundary_position"], groups[-1][-1]["boundary_position"]
            pos = c["boundary_position"]
            if utt_chars[last:pos].sum() <= gap_chars and utt_chars[first:pos].sum() <= max_span_chars:
                groups[-1].append(c)
                continue
        groups.append([c])
    return groups


def build_span(span_id, group, utt_chars):
    start, end = group[0]["boundary_position"], group[-1]["boundary_position"]
    best = max(group, key=lambda c: c["depth_score"])
    return {
        "span_id": span_id,
        "span_start": start,                    # [span_start]~[span_end] 사이 어딘가에서 새 주제가 시작된다고 보는 구간
        "span_end": end,
        "span_chars": int(utt_chars[start:end].sum()),
        "member_candidate_ids": [c["candidate_id"] for c in group],
        "member_positions": [c["boundary_position"] for c in group],
        "boundary_position": best["boundary_position"],   # 대표 경계 (구간 안 depth 최대 후보)
        "depth_score": best["depth_score"],
    }


def split_segments(utterances, spans, utt_chars):
    # 대표 경계 b마다 [이전 경계, b-1] / [b, 다음 경계-1] ... 로 나눔 (첫 구간은 0번 발화부터)
    starts = [0] + [s["boundary_position"] for s in spans if s["boundary_position"] > 0]
    ends = starts[1:] + [len(utterances)]
    return [{
        "segment_id": i,
        "utterance_start": start,               # 포함
        "utterance_end": end - 1,               # 포함
        "num_utterances": end - start,
        "char_count": int(utt_chars[start:end].sum()),
        "utterances": utterances[start:end],
    } for i, (start, end) in enumerate(zip(starts, ends))]

In [ ]:
# --------------------------------------------------------------------------------------------------
# 6. 전체 실행: 전사문 → 청킹 → 임베딩 → 경계 후보 탐지 → 구간 병합 → 회의록 분할 → 결과 JSON 구성
# --------------------------------------------------------------------------------------------------
import pandas as pd


def detect_boundary_candidates(input_path):
    transcript = load_transcript(input_path)
    utterances = transcript["utterances"]
    texts = [u["text"] for u in utterances]
    utt_chars = np.array([len(t) for t in texts])

    # 청킹 → 임베딩
    chunks = chunk_utterances(texts)
    chunk_emb = pooled_chunk_embeddings(chunks, embed_utterances(texts), utt_chars)

    # 경계 후보 탐지
    sims = block_similarities(chunk_emb)
    depths = depth_scores(sims)
    gaps, threshold = select_candidates(depths)
    candidates = [{
        "candidate_id": i,
        "boundary_position": boundary_position(chunks, g),   # 이 발화부터 새 주제라고 보는 후보 위치
        "gap_index": g,                                       # chunk g | chunk g+1
        "similarity": round(float(sims[g]), 4),
        "depth_score": round(float(depths[g]), 4),
    } for i, g in enumerate(gaps)]

    # 인접 후보 구간 병합 → 대표 경계로 회의록 분할
    groups = merge_candidates(candidates, utt_chars)
    spans = [build_span(i, group, utt_chars) for i, group in enumerate(groups)]
    segments = split_segments(utterances, spans, utt_chars)

    return {
        "meeting_id": transcript["meeting_id"],
        "metadata": transcript["metadata"],
        "chunking": {
            "unit": "chars",
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "num_chunks": len(chunks),
        },
        "chunks": chunks,
        "detection": {
            "embedding_model": EMBEDDING_MODEL,
            "embedding_mode": "pooled",
            "filler_max_chars": FILLER_MAX_CHARS,
            "block_size": BLOCK_SIZE,
            "score": "depth score of block cosine similarity (TextTiling)",
            "depth_std_k": DEPTH_STD_K,
            "depth_threshold": round(threshold, 4),
            "merge_gap_chars": MERGE_GAP_CHARS,
            "merge_max_span_chars": MERGE_MAX_SPAN_CHARS,
            "num_utterances": len(utterances),
            "num_gaps": len(sims),
            "num_candidates": len(candidates),
            "num_spans": len(spans),
            "num_segments": len(segments),
        },
        "candidates": candidates,
        "spans": spans,
        "segments": segments,
    }


result = detect_boundary_candidates(INPUT_PATH)
d = result["detection"]

print(f"[{result['meeting_id']}] 발화 {d['num_utterances']}개 → 청크 {result['chunking']['num_chunks']}개 "
      f"→ 경계 후보 {d['num_candidates']}개 → 병합 구간 {d['num_spans']}개 → 회의록 {d['num_segments']}개 구간으로 분할")
display(pd.DataFrame([{
    "병합 구간 번호": s["span_id"],
    "구간": f"{s['span_start']}" if s["span_start"] == s["span_end"] else f"{s['span_start']}~{s['span_end']}",
    "묶인 후보": s["member_positions"],
    "대표 경계": s["boundary_position"],
    "depth": s["depth_score"],
} for s in result["spans"]]).set_index("병합 구간 번호"))
display(pd.DataFrame([{
    "분할 구간 번호": seg["segment_id"],
    "발화 범위": f"{seg['utterance_start']}~{seg['utterance_end']}",
    "발화 수": seg["num_utterances"],
    "글자 수": seg["char_count"],
    "첫 발화": seg["utterances"][0]["text"][:40],
} for seg in result["segments"]]).set_index("분할 구간 번호"))

In [ ]:
# --------------------------------------------------------------------------------------------------
# 7. 결과 저장 → OUTPUT_DIR/{meeting_id}_boundary_candidates.json
# --------------------------------------------------------------------------------------------------
if SAVE_RESULT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / f"{result['meeting_id']}_boundary_candidates.json"
    output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print("저장 완료:", output_path)
else:
    print("SAVE_RESULT = False → 저장하지 않음")